In [3]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.impute import KNNImputer

df = pd.read_csv(r'C:\Users\Dhruv stark\Desktop\machine learning practice\spaceship\train.csv')
df.head()
df.isnull().sum()
df.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name', 'Transported'],
      dtype='object')

In [5]:

# Numeric columns only for KNN
numeric_col = df[['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']]
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
imputer = IterativeImputer(random_state=42,max_iter=10)
df[['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']] = imputer.fit_transform(numeric_col)

# Categorical columns separately
cat_cols = ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'VIP','Name']
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

df.isnull().sum()


PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

In [25]:
df.columns
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,8693.0,28.869159,14.368427,0.0,20.0,27.0,37.0,79.0
RoomService,8693.0,223.937375,662.046544,0.0,0.0,0.0,51.0,14327.0
FoodCourt,8693.0,452.891982,1597.230255,0.0,0.0,0.0,81.0,29813.0
ShoppingMall,8693.0,172.321477,598.500190,0.0,0.0,0.0,30.0,23492.0
Spa,8693.0,308.054527,1126.161421,0.0,0.0,0.0,62.0,22408.0
VRDeck,8693.0,301.593926,1134.654404,0.0,0.0,0.0,49.0,24133.0


In [10]:
from sklearn.model_selection import train_test_split
df.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [11]:
df.columns

Index(['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age',
       'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'Name', 'Transported'],
      dtype='object')

In [31]:
X = df.drop(columns=['PassengerId', 'Name'])
Y = df['Transported']

In [17]:
X.head()

,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


In [18]:
Y.head()

0    False
1     True
2    False
3    False
4     True
Name: Transported, dtype: bool

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [27]:
df.shape

(8693, 14)

In [38]:
from sklearn.model_selection import KFold,cross_val_score,StratifiedKFold 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X = df.drop(columns=['PassengerId', 'Name'])
Y = df['Transported']

x_train,x_test,y_train,y_test = train_test_split(X,Y,test_size=0.2, stratify=Y,random_state=42)
x_train.shape,x_test.shape

numeric_cols = x_train.select_dtypes(include='number').columns.tolist()

# Step 1 — define model FIRST
model = LogisticRegression(max_iter=1000)
skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model,x_train[numeric_cols], y_train, cv=skfold)
print(f'Score: {scores} ± {scores.std():.4f}')


#

Score: [0.78864127 0.77426312 0.7721064  0.75844716 0.78201439] ± 0.0102


## professionally we make 

In [33]:
x_train.value_counts()


HomePlanet  CryoSleep  Cabin     Destination  Age   VIP    RoomService  FoodCourt  ShoppingMall  Spa  VRDeck  Transported
Europa      True       G/734/S   55 Cancri e  23.0  False  0.0          0.0        0.0           0.0  0.0     True           3
Earth       False      G/734/S   TRAPPIST-1e  7.0   False  0.0          0.0        0.0           0.0  0.0     False          2
            True       G/730/S   TRAPPIST-1e  2.0   False  0.0          0.0        0.0           0.0  0.0     True           2
                       G/450/S   55 Cancri e  3.0   False  0.0          0.0        0.0           0.0  0.0     True           2
Mars        False      F/1787/P  TRAPPIST-1e  1.0   False  0.0          0.0        0.0           0.0  0.0     True           2
                                                                                                                            ..
Earth       False      G/574/P   TRAPPIST-1e  1.0   False  0.0          0.0        0.0           0.0  0.0     True  

## Is stratify it really good for model?
Yes — but only for classification. Not for regression.
SituationUse stratify?
Classification (True/False, categories)✅ Always
Imbalanced classes (fraud, disease)✅ Critical
Regression (predicting a number)❌ No — doesn't apply
Your Spaceship Titanic dataset✅ Yes

## MOST PROFRESSIONAL USED THE KFOLD AND IT TWO TYPE REGULARIZED AND STRATIFIED 

K-Fold Cross Validation is a technique used to evaluate Machine Learning models more reliably.

Instead of:

training once
testing once

the model trains and tests multiple times on different parts of the dataset.